In [2]:
pip install -q google-genai pandas-gbq

In [3]:
from google import genai
client = genai.Client(api_key="YOUR_GEMINI_API_KEY")

result = client.models.embed_content(
    model="models/gemini-embedding-001",
    contents="Crime rate in the Austin neighborhood of Chicago"
)

embedding = result.embeddings[0].values
print(f"Length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Length: 3072
First 5 values: [-0.0030353323, -0.0033826898, 0.008175602, -0.02764508, -0.0035436938]


In [4]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

e1 = client.models.embed_content(model="models/gemini-embedding-001", contents="violent crime in Chicago").embeddings[0].values
e2 = client.models.embed_content(model="models/gemini-embedding-001", contents="robbery and assault in Illinois").embeddings[0].values
e3 = client.models.embed_content(model="models/gemini-embedding-001", contents="best pasta recipe").embeddings[0].values

print("Crime vs Crime:", cosine_similarity(e1, e2))
print("Crime vs Recipe:", cosine_similarity(e1, e3))

Crime vs Crime: 0.7017284985515155
Crime vs Recipe: 0.45693921382065


In [5]:
from google.colab import auth
auth.authenticate_user()

In [6]:
from google.cloud import bigquery
client_bq = bigquery.Client(project="project-72181980-533d-4978-aa7")

query = """
SELECT * FROM `project-72181980-533d-4978-aa7.de_practice.chicago_crime_clean`
LIMIT 3
"""

df = client_bq.query(query).to_dataframe()
print(df.columns.tolist())
print(df.head(3))

['unique_key', 'date', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'year', 'latitude', 'longitude', 'month', 'hour', 'is_night', 'is_arrested']
   unique_key                date                      primary_type  \
0    11985355 2020-02-19 00:18:00  INTERFERENCE WITH PUBLIC OFFICER   
1    12239534 2020-12-05 00:00:00           CRIMINAL SEXUAL ASSAULT   
2    11969243 2020-02-02 00:45:00                 WEAPONS VIOLATION   

                      description            location_description  arrest  \
0  RESIST/OBSTRUCT/DISARM OFFICER                    CTA PLATFORM    True   
1                  NON-AGGRAVATED                       APARTMENT   False   
2        UNLAWFUL POSS OF HANDGUN  PARKING LOT/GARAGE(NON.RESID.)    True   

   domestic  year   latitude  longitude  month  hour  is_night  is_arrested  
0     False  2020  41.868165 -87.627440      2     0      True            1  
1     False  2020  41.777558 -87.615558     12     0      True           

In [7]:
query = """
SELECT unique_key, primary_type, description, location_description, year, latitude, longitude
FROM `project-72181980-533d-4978-aa7.de_practice.chicago_crime_clean`
LIMIT 500
"""

df = client_bq.query(query).to_dataframe()
print(df.shape)
print(df.head(3))

(500, 7)
   unique_key                      primary_type  \
0    11985355  INTERFERENCE WITH PUBLIC OFFICER   
1    12239534           CRIMINAL SEXUAL ASSAULT   
2    11969243                 WEAPONS VIOLATION   

                      description            location_description  year  \
0  RESIST/OBSTRUCT/DISARM OFFICER                    CTA PLATFORM  2020   
1                  NON-AGGRAVATED                       APARTMENT  2020   
2        UNLAWFUL POSS OF HANDGUN  PARKING LOT/GARAGE(NON.RESID.)  2020   

    latitude  longitude  
0  41.868165 -87.627440  
1  41.777558 -87.615558  
2  41.776847 -87.617249  


In [8]:
def get_embedding(text):
  result = client.models.embed_content(
      model="models/gemini-embedding-001",
      contents=text
    )
  return result.embeddings[0].values

df['text'] = df['primary_type'] + " - " + df['description'] + " at " + df['location_description']

print("Sample sentences:")
print(df['text'][0])
print(df['text'][1])
print(df['text'][2])

Sample sentences:
INTERFERENCE WITH PUBLIC OFFICER - RESIST/OBSTRUCT/DISARM OFFICER at CTA PLATFORM
CRIMINAL SEXUAL ASSAULT - NON-AGGRAVATED at APARTMENT
WEAPONS VIOLATION - UNLAWFUL POSS OF HANDGUN at PARKING LOT/GARAGE(NON.RESID.)


In [9]:
import time

embeddings = []

for i , text in enumerate(df['text']):
  while True:
        try:
            embedding = get_embedding(text)
            embeddings.append(embedding)
            time.sleep(0.7)  # increased delay
            break
        except Exception as e:
            print(f"Rate limit hit at row {i}, waiting 60 seconds...")
            time.sleep(60)  # wait 1 minute then retry


  if i % 50 == 0:
        print(f"Done {i}/500 rows...")

df['embedding'] = embeddings
print("All done!")
print(df.shape)

Done 0/500 rows...
Done 50/500 rows...
Done 100/500 rows...
Done 150/500 rows...
Done 200/500 rows...
Done 250/500 rows...
Done 300/500 rows...
Done 350/500 rows...
Done 400/500 rows...
Done 450/500 rows...
All done!
(500, 9)


In [10]:
from google.cloud import bigquery
import json

# Convert embedding list to JSON string (BQ stores it as STRING)
df['embedding'] = df['embedding'].apply(lambda x: json.dumps(x))

# Write to BigQuery
table_id = "project-72181980-533d-4978-aa7.de_practice.chicago_crime_embeddings"

df.to_gbq(
    destination_table="de_practice.chicago_crime_embeddings",
    project_id="project-72181980-533d-4978-aa7",
    if_exists="replace"
)

print("Written to BigQuery successfully!")

/tmp/ipykernel_686/2895350913.py:10: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 7530.17it/s]

Written to BigQuery successfully!


In [11]:
query = """
SELECT unique_key, primary_type, description, location_description, year, text, embedding
FROM `project-72181980-533d-4978-aa7.de_practice.chicago_crime_embeddings`
LIMIT 5
"""

df_embeddings = client_bq.query(query).to_dataframe()
print(df_embeddings.shape)
print(df_embeddings.columns.tolist())


(5, 7)
['unique_key', 'primary_type', 'description', 'location_description', 'year', 'text', 'embedding']


In [14]:
import json

def search_crimes(user_question , top_K =5):
  q_embedding = client.models.embed_content(
       model="models/gemini-embedding-001",
       contents=user_question
  ).embeddings[0].values


  query = """
    SELECT unique_key, primary_type, description, location_description, year, text, embedding
    FROM `project-72181980-533d-4978-aa7.de_practice.chicago_crime_embeddings`
    """

  client_bq.query(query).to_dataframe()


  df['similarity'] = df['embedding'].apply(
        lambda x: cosine_similarity(q_embedding, json.loads(x))
    )

  top_results = df.sort_values('similarity' , ascending=False).head(top_K)

  return top_results[['primary_type', 'description', 'location_description', 'year', 'similarity']]

# Test it

results = search_crimes("gun crimes at night in residential areas")
print(results)

          primary_type                     description  \
465  WEAPONS VIOLATION            UNLAWFUL USE HANDGUN   
353      OTHER OFFENSE      OTHER CRIME AGAINST PERSON   
376            ASSAULT             AGGRAVATED: HANDGUN   
106      OTHER OFFENSE  OTHER CRIME INVOLVING PROPERTY   
13       OTHER OFFENSE  OTHER CRIME INVOLVING PROPERTY   

              location_description  year  similarity  
465  RESIDENTIAL YARD (FRONT/BACK)  2020    0.708153  
353                      RESIDENCE  2020    0.699603  
376                      RESIDENCE  2020    0.699332  
106                      RESIDENCE  2020    0.689937  
13                       RESIDENCE  2020    0.689937  
